In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import numpy as np
from typing import Dict, Tuple, List

# 環境設定 (仮の値を設定。実際の環境に合わせてください)
GRID_SIZE = 10
NUM_AGENTS = 2
NUM_ORDERS = 3
DROPOFF_LOCATION = (5, 5) # 以前のコードで定義された変数を参照

# --- 1-1. MLPAgent (RNNの代わり) ---
class MLPAgent(nn.Module):
    """RNNを使用しない、よりシンプルなエージェントネットワーク"""
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(MLPAgent, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        # 隠れ状態 h は不要なため、Q値のみを返す
        return self.net(x)

# --- 1-2. QMixer (ハイパーネットワークはそのまま使用) ---
# QMixerの定義は前回のDeepQMixerのバージョンを流用することを想定。
# (ここではコードは省略しますが、必ず実装してください)
# from DeepQMixer import DeepQMixer as QMixer

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from typing import Dict, Tuple, List
import time
import numpy as np
from IPython import display # Jupyter環境用（オプション）

# 環境設定
GRID_SIZE = 10
NUM_AGENTS = 2
NUM_ORDERS = 3
PICKUP_LOCATIONS = [(1, 1), (8, 1), (5, 8)]
DROPOFF_LOCATION = (5, 5)

class WarehouseEnv:
    def __init__(self, size: int = GRID_SIZE, num_agents: int = NUM_AGENTS):
        self.size = size
        self.num_agents = num_agents
        self.action_space = 5  # 0:待機, 1:上, 2:下, 3:左, 4:右

        # Matplotlib用の図の保持
        self.fig = None
        self.ax = None

        # 状態の初期化
        self.reset()

    def reset(self) -> Dict[int, Tuple]:
        """環境を初期化し、初期状態を返します。"""
        self.agent_positions: Dict[int, Tuple[int, int]] = {
            i: (random.randint(0, self.size - 1), random.randint(0, self.size - 1))
            for i in range(self.num_agents)
        }
        self.agent_holding: Dict[int, bool] = {i: False for i in range(self.num_agents)}
        self.remaining_orders: List[int] = list(range(NUM_ORDERS))
        return self._get_obs()

    def _get_obs(self) -> Dict[int, Tuple]:
        obs = {}
        for i in range(self.num_agents):
            obs[i] = (
                self.agent_positions[i],
                self.agent_holding[i],
                tuple(self.remaining_orders)
            )
        return obs

    def step(self, actions: Dict[int, int]) -> Tuple[Dict, Dict, Dict, Dict]:
        next_positions: Dict[int, Tuple[int, int]] = {}
        rewards: Dict[int, float] = {i: -0.1 for i in range(self.num_agents)}

        # 1. 位置の更新
        for i, action in actions.items():
            current_x, current_y = self.agent_positions[i]
            next_x, next_y = current_x, current_y

            if action == 1: next_y += 1  # 上
            elif action == 2: next_y -= 1  # 下
            elif action == 3: next_x -= 1  # 左
            elif action == 4: next_x += 1  # 右

            next_x = np.clip(next_x, 0, self.size - 1)
            next_y = np.clip(next_y, 0, self.size - 1)
            next_positions[i] = (next_x, next_y)

        # 2. 衝突判定
        final_positions = self.agent_positions.copy()
        is_collision = False

        for i in range(self.num_agents):
            pos = next_positions[i]
            is_colliding = False
            for j in range(self.num_agents):
                if i != j and pos == next_positions[j]:
                    is_colliding = True
                    break
            if is_colliding:
                rewards[i] -= 5.0
                is_collision = True
            else:
                final_positions[i] = pos

        self.agent_positions = final_positions

        # 3. ピックアップ・ドロップオフ
        for i in range(self.num_agents):
            current_pos = self.agent_positions[i]
            if not self.agent_holding[i]:
                for order_idx in self.remaining_orders:
                    if current_pos == PICKUP_LOCATIONS[order_idx]:
                        self.agent_holding[i] = True
                        self.remaining_orders.remove(order_idx)
                        rewards[i] += 10.0
                        break
            elif self.agent_holding[i]:
                if current_pos == DROPOFF_LOCATION:
                    self.agent_holding[i] = False
                    rewards[i] += 50.0

        done = {i: len(self.remaining_orders) == 0 for i in range(self.num_agents)}
        return self._get_obs(), rewards, done, {"collision": is_collision}

    # --- 追加された可視化メソッド ---
    def render(self, mode='text', sleep_time=0.5):
        """
        環境を可視化します。
        mode='text': コンソールに文字で表示
        mode='graphic': Matplotlibで図として表示
        """
        if mode == 'text':
            self._render_text()
        elif mode == 'graphic':
            self._render_graphic(sleep_time)

    def _render_text(self):
        grid = [['.' for _ in range(self.size)] for _ in range(self.size)]

        # 場所のマーク (y座標は下から上へ増えるため、表示時は反転させるか注意が必要)
        # ここでは (0,0) を左下として扱います
        x, y = DROPOFF_LOCATION
        grid[self.size - 1 - y][x] = 'D'  # Dropoff

        for idx in self.remaining_orders:
            x, y = PICKUP_LOCATIONS[idx]
            grid[self.size - 1 - y][x] = 'P'  # Pickup

        for i, pos in self.agent_positions.items():
            x, y = pos
            char = f'A{i}'
            if self.agent_holding[i]:
                char = f'H{i}' # Holding

            # 同じ場所に重なった場合の表示処理（簡易）
            if grid[self.size - 1 - y][x] not in ['.', 'P', 'D']:
                grid[self.size - 1 - y][x] += char
            else:
                grid[self.size - 1 - y][x] = char

        print("-" * (self.size * 3))
        for row in grid:
            print(" ".join([f"{c:>2}" for c in row]))
        print("-" * (self.size * 3))

    # --- WarehouseEnv クラス内の _render_graphic メソッドの修正 ---
    # ⚠️ 注意: これは WarehouseEnv クラスの内部にあると仮定
    def _render_graphic(self, sleep_time):
        # sleep_time はここでは完全に無視される（run_learned_agent側で処理）

        if self.fig is None:
            plt.ioff() # インタラクティブモードをオフにする（描画更新はdisplayに任せる）
            self.fig, self.ax = plt.subplots(figsize=(6, 6))

        self.ax.clear()
        self.ax.set_xlim(-0.5, self.size - 0.5)
        self.ax.set_ylim(-0.5, self.size - 0.5)
        self.ax.set_xticks(range(self.size))
        self.ax.set_yticks(range(self.size))
        self.ax.grid(True)
        self.ax.set_title(f"Orders Remaining: {len(self.remaining_orders)}")

        # ドロップオフ地点 (赤色)
        dx, dy = DROPOFF_LOCATION
        self.ax.add_patch(patches.Rectangle((dx-0.5, dy-0.5), 1, 1, color='red', alpha=0.3, label='Dropoff'))
        self.ax.text(dx, dy, 'Drop', ha='center', va='center', fontsize=8, color='darkred')

        # ピックアップ地点 (青色)
        for idx in self.remaining_orders:
            px, py = PICKUP_LOCATIONS[idx]
            self.ax.add_patch(patches.Rectangle((px-0.5, py-0.5), 1, 1, color='blue', alpha=0.3, label='Pickup'))
            self.ax.text(px, py, 'Pick', ha='center', va='center', fontsize=8, color='darkblue')

        # エージェント (円)
        colors = ['green', 'orange', 'purple', 'cyan']
        for i, pos in self.agent_positions.items():
            ax, ay = pos
            color = colors[i % len(colors)]
            edgecolor = 'black'
            linewidth = 1
            if self.agent_holding[i]:
                linewidth = 3
                edgecolor = 'red' # 荷物持ち強調

            circle = patches.Circle((ax, ay), 0.3, facecolor=color, edgecolor=edgecolor, linewidth=linewidth, label=f'Agent {i}')
            self.ax.add_patch(circle)
            self.ax.text(ax, ay, f'A{i}', ha='center', va='center', color='white', fontweight='bold')

        # 🚨 描画更新と待機を削除！ (run_learned_agent側で処理する)
        # plt.draw()
        # if sleep_time > 0.0:
        #   plt.pause(sleep_time)

    # def render_frame(self):
    #     fig, ax = plt.subplots(figsize=(6, 6))

    #     ax.set_xlim(-0.5, self.size - 0.5)
    #     ax.set_ylim(-0.5, self.size - 0.5)
    #     ax.set_xticks(range(self.size))
    #     ax.set_yticks(range(self.size))
    #     ax.grid(True)

    #     # Dropoff
    #     dx, dy = DROPOFF_LOCATION
    #     ax.add_patch(patches.Rectangle((dx-0.5, dy-0.5), 1, 1, color='red', alpha=0.3))

    #     # Pickup
    #     for idx in self.remaining_orders:
    #         px, py = PICKUP_LOCATIONS[idx]
    #         ax.add_patch(patches.Rectangle((px-0.5, py-0.5), 1, 1, color='blue', alpha=0.3))

    #     # Agents
    #     colors = ['green', 'orange', 'purple', 'cyan']
    #     for i, (x, y) in self.agent_positions.items():
    #         circle = patches.Circle((x, y), 0.3, color=colors[i % len(colors)])
    #         ax.add_patch(circle)
    #         # 荷物を持っているかどうかの表示を追加すると分かりやすいです
    #         text = f'A{i}'
    #         if self.agent_holding[i]:
    #             text += '*' # 荷物持ちマーク
    #         ax.text(x, y, text, ha='center', va='center', color='white')

    #     # --- 修正箇所: 画像データへの変換 ---
    #     fig.canvas.draw()

    #     # buffer_rgba() を使用して RGBA データを取得
    #     # np.asarray で NumPy 配列に変換
    #     image = np.asarray(fig.canvas.buffer_rgba())

    #     # アルファチャンネル (透明度) を削除して RGB 画像にする (必要であれば)
    #     image = image[:, :, :3]

    #     plt.close(fig)
    #     return image

    def render_frame(self):
        fig, ax = plt.subplots(figsize=(6, 6))

        ax.set_xlim(-0.5, self.size - 0.5)
        ax.set_ylim(-0.5, self.size - 0.5)
        ax.set_xticks(range(self.size))
        ax.set_yticks(range(self.size))
        ax.grid(True)

        # Dropoff
        dx, dy = DROPOFF_LOCATION
        ax.add_patch(patches.Rectangle((dx-0.5, dy-0.5), 1, 1, color='red', alpha=0.3))

        # Pickup
        for idx in self.remaining_orders:
            px, py = PICKUP_LOCATIONS[idx]
            ax.add_patch(patches.Rectangle((px-0.5, py-0.5), 1, 1, color='blue', alpha=0.3))

        # Agents
        colors = ['green', 'orange', 'purple', 'cyan']
        for i, (x, y) in self.agent_positions.items():
            circle = patches.Circle((x, y), 0.3, color=colors[i % len(colors)])
            ax.add_patch(circle)
            ax.text(x, y, f'A{i}', ha='center', va='center', color='white')

        fig.canvas.draw()
        image = np.asarray(fig.canvas.buffer_rgba())[..., :3]

        plt.close(fig)
        return image


In [ ]:
class QMixer(nn.Module):
    def __init__(self, n_agents, state_shape, mixing_embed_dim, hypernet_embed_dim):
        super().__init__()
        self.n_agents = n_agents
        self.mixing_embed_dim = mixing_embed_dim
        # 1. ハイパーネットワークの定義
        # 環境状態 (state) を入力とし、Mixing Networkの重みW1を生成
        self.hyper_w1 = nn.Sequential(
            nn.Linear(state_shape, hypernet_embed_dim),
            nn.ReLU(),
            nn.Linear(hypernet_embed_dim, mixing_embed_dim * n_agents)
        )

        # バイアスb1も環境状態から生成
        self.hyper_b1 = nn.Linear(state_shape, mixing_embed_dim)

        # W2 (Mixing Networkの2層目の重み) を生成
        self.hyper_w2 = nn.Sequential(
            nn.Linear(state_shape, hypernet_embed_dim),
            nn.ReLU(),
            nn.Linear(hypernet_embed_dim, mixing_embed_dim)
        )

        # バイアスb2 (最終出力層のバイアス) を環境状態から生成
        self.hyper_b2 = nn.Sequential(
            nn.Linear(state_shape, hypernet_embed_dim),
            nn.ReLU(),
            nn.Linear(hypernet_embed_dim, 1)
        )

        # 2. 最終出力層（ダミー）
        # 実際にはハイパーネットワークが生成した重みで演算するが、サイズ調整のために定義
        self.V = nn.Sequential(nn.Linear(state_shape, mixing_embed_dim), nn.ReLU(), nn.Linear(mixing_embed_dim, 1))

    def forward(self, agent_qs, states):
        # agent_qs: 全エージェントのQ値 (batch_size, n_agents)
        # states: 環境の全体状態 (batch_size, state_shape)

        bs = agent_qs.size(0)

        # 1. 隠れ層 W1 の計算 (重みの生成と非負制約)
        W1 = self.hyper_w1(states).view(bs, self.n_agents, self.mixing_embed_dim)
        # 非負制約: 重みをReLUに通す
        W1 = F.relu(W1)

        # 2. 隠れ層 B1 (バイアス) の計算
        B1 = self.hyper_b1(states).view(bs, 1, self.mixing_embed_dim)

        # 3. 第1層の計算: (Q_i * W1) + B1
        # agent_qs: (bs, 1, n_agents), W1: (bs, n_agents, mixing_embed_dim)
        hidden = torch.bmm(agent_qs.unsqueeze(1), W1)
        # hidden: (bs, 1, mixing_embed_dim)
        hidden = F.relu(hidden + B1)

        # 4. 出力層 W2 の計算 (重みの生成と非負制約)
        W2 = self.hyper_w2(states).view(bs, self.mixing_embed_dim, 1)
        # 非負制約: 重みをReLUに通す
        W2 = F.relu(W2)

        # 5. 出力層 B2 (バイアス) の計算
        B2 = self.hyper_b2(states).view(bs, 1, 1)

        # 6. 第2層の計算: (hidden * W2) + B2
        # V(s)項 (全エージェントに共通のバイアス項)
        v = self.V(states).view(bs, 1, 1)

        # 最終的なQ_totの出力
        q_tot = torch.bmm(hidden, W2) + B2 + v
        # q_tot: (batch_size, 1, 1)
        return q_tot.squeeze()

#

In [ ]:
# class IntegratedQMixAgent: ... (前回の修正版クラス定義をここに配置)
# (簡略化のため、init_hiddenなどのRNN関連メソッドは削除)
class IntegratedQMixAgent:
    def __init__(self, env, obs_shape, state_shape, n_actions, lr=5e-4, gamma=0.99, mixing_embed_dim=32, hidden_dim=64):
        memory_capacity = 5000
        self.env = env
        self.n_agents = env.num_agents
        self.n_actions = n_actions
        self.gamma = gamma
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.memory = QMixReplayMemory(memory_capacity)

        self.agent_net = MLPAgent(obs_shape, hidden_dim, n_actions).to(self.device)
        self.mixer_net = QMixer(self.n_agents, state_shape, mixing_embed_dim, hypernet_embed_dim=hidden_dim).to(self.device)

        self.target_agent_net = MLPAgent(obs_shape, hidden_dim, n_actions).to(self.device)
        self.target_mixer_net = QMixer(self.n_agents, state_shape, mixing_embed_dim, hypernet_embed_dim=hidden_dim).to(self.device)

        self.target_agent_net.load_state_dict(self.agent_net.state_dict())
        self.target_mixer_net.load_state_dict(self.mixer_net.state_dict())

        params = list(self.agent_net.parameters()) + list(self.mixer_net.parameters())
        self.optimizer = torch.optim.Adam(params, lr=lr)

    def _obs_to_tensor(self, obs: Dict[int, Tuple], is_state: bool = False):
        if is_state:
            state_vec = []
            for i in range(self.n_agents):
                state_vec.extend([x / (GRID_SIZE - 1) for x in obs[i][0]])
                state_vec.append(1.0 if obs[i][1] else 0.0)

            remaining_orders_set = set(obs[0][2])
            for order_idx in range(NUM_ORDERS):
                state_vec.append(1.0 if order_idx in remaining_orders_set else 0.0)

            return torch.FloatTensor(state_vec).to(self.device).unsqueeze(0)
        else:
            tensors = {}
            for i in range(self.n_agents):
                obs_i = [x / (GRID_SIZE - 1) for x in obs[i][0]]
                obs_i.append(1.0 if obs[i][1] else 0.0)

                agent_id_vec = [0.0] * self.n_agents
                agent_id_vec[i] = 1.0
                obs_i.extend(agent_id_vec)

                tensors[i] = torch.FloatTensor(obs_i).to(self.device).unsqueeze(0)
            return tensors

    def get_actions(self, obs: Dict[int, Tuple], epsilon: float) -> Dict[int, int]:
        actions = {}
        if random.random() < epsilon:
            for i in range(self.n_agents):
                actions[i] = random.randint(0, self.n_actions - 1)
        else:
            agent_obs_tensors = self._obs_to_tensor(obs, is_state=False)
            with torch.no_grad():
                for i in range(self.n_agents):
                    q_values = self.agent_net(agent_obs_tensors[i])
                    actions[i] = q_values.max(dim=-1)[1].item()
        return actions

    def learn(self, batch, target_update_interval, update_counter):
        current_state = torch.cat([self._obs_to_tensor(t[0], is_state=True) for t in batch], dim=0)
        next_state = torch.cat([self._obs_to_tensor(t[3], is_state=True) for t in batch], dim=0)
        rewards = torch.FloatTensor([sum(t[2].values()) for t in batch]).to(self.device).unsqueeze(1)
        terminated = torch.FloatTensor([all(t[4].values()) for t in batch]).to(self.device).unsqueeze(1)
        actions_batch = torch.LongTensor([[t[1][i] for i in range(self.n_agents)] for t in batch]).to(self.device)
        obs_batch = [torch.cat([self._obs_to_tensor(t[0], is_state=False)[i] for t in batch], dim=0) for i in range(self.n_agents)]
        next_obs_batch = [torch.cat([self._obs_to_tensor(t[3], is_state=False)[i] for t in batch], dim=0) for i in range(self.n_agents)]

        # 1. 現在のQ_totの計算
        agent_qs = []
        for i in range(self.n_agents):
            q_vals = self.agent_net(obs_batch[i])
            chosen_q = torch.gather(q_vals, dim=1, index=actions_batch[:, i].unsqueeze(1))
            agent_qs.append(chosen_q)

        agent_qs = torch.cat(agent_qs, dim=1)
        q_tot = self.mixer_net(agent_qs, current_state)

        # 2. TDターゲットの計算 (Double DQN 適用)
        target_agent_qs = []
        with torch.no_grad():
            for i in range(self.n_agents):
                next_q_online = self.agent_net(next_obs_batch[i])
                next_action_argmax = next_q_online.max(dim=1)[1].unsqueeze(1)

                target_q_vals = self.target_agent_net(next_obs_batch[i])

                target_max_q = target_q_vals.gather(dim=1, index=next_action_argmax)
                target_agent_qs.append(target_max_q)

            target_agent_qs = torch.cat(target_agent_qs, dim=1)
            target_q_tot = self.target_mixer_net(target_agent_qs, next_state)

        td_target = rewards + self.gamma * target_q_tot * (1 - terminated)

        # 3. 損失の計算と最適化
        loss = F.mse_loss(q_tot, td_target.detach())

        loss.backward()

        torch.nn.utils.clip_grad_norm_(self.agent_net.parameters(), 10)
        torch.nn.utils.clip_grad_norm_(self.mixer_net.parameters(), 10)

        self.optimizer.step()

        if update_counter % target_update_interval == 0:
            self.update_target_networks()

        return loss.item()

    def update_target_networks(self):
        self.target_agent_net.load_state_dict(self.agent_net.state_dict())
        self.target_mixer_net.load_state_dict(self.mixer_net.state_dict())

In [ ]:
from collections import deque
import random

class QMixReplayMemory:
    def __init__(self, capacity):
        self.memory = deque(maxlen=capacity)
    def push(self, state, action, next_state, reward, done):
        self.memory.append((state, action, next_state, reward, done))
    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)
    def __len__(self):
        return len(self.memory)

#

In [ ]:
# --- 学習パラメータ ---
# 環境パラメータ (以前の定義に合わせてください)
GRID_SIZE = 10
NUM_AGENTS = 2
NUM_ORDERS = 3
PICKUP_LOCATIONS = [(1, 1), (8, 1), (5, 8)]
DROPOFF_LOCATION = (5, 5)

# QMIX学習パラメータ
BATCH_SIZE = 128
GAMMA = 0.99
EPS_START = 1.0
EPS_END = 0.05
EPS_DECAY = 50000        # 減衰をゆっくりにする (ステップ数ベース)
TARGET_UPDATE_INTERVAL = 200 # ターゲットネットワーク更新頻度 (ステップ数)
MEMORY_SIZE = 50000
NUM_EPISODES = 5000      # 実行するエピソード数
MAX_STEPS_PER_EPISODE = 200
LEARNING_FREQ = 4        # 4ステップごとに1回学習

# 環境・エージェントの形状定義
ACTION_SPACE = 5         # 5: 停止, 上, 下, 左, 右
OBS_SHAPE = 2 + 1 + NUM_AGENTS # 位置(2) + 荷物(1) + Agent ID(2) = 5
STATE_SHAPE = (2 + 1) * NUM_AGENTS + NUM_ORDERS # 位置(2) + 荷物(1)) * 2 + 注文(3)

total_steps = 0
episode_rewards = []
losses = []

In [ ]:
def run_qmix_training():
    env = WarehouseEnv(GRID_SIZE, NUM_AGENTS)
    replay_buffer = QMixReplayMemory(MEMORY_SIZE)
    agent = IntegratedQMixAgent(env, OBS_SHAPE, STATE_SHAPE, ACTION_SPACE, gamma=GAMMA)

    total_steps = 0
    rewards_history = []
    losses_history = []

    print(f"--- QMIX Learning Start --- (Agents: {NUM_AGENTS}, Grid: {GRID_SIZE}x{GRID_SIZE})")

    for i_episode in range(1, NUM_EPISODES + 1):
        obs = env.reset()
        episode_reward = 0
        done_flag = False

        for t in range(MAX_STEPS_PER_EPISODE):
            # 1. ε-greedy法による行動選択
            epsilon = EPS_END + (EPS_START - EPS_END) * np.exp(-1. * total_steps / EPS_DECAY)
            actions = agent.get_actions(obs, epsilon)

            # 2. 環境ステップ
            next_obs, rewards, done, info = env.step(actions)

            # チーム全体の終了フラグ
            terminated_flag = all(done.values())

            # 3. リプレイバッファに保存
            # リプレイメモリに保存されるデータは (obs, actions, next_obs, rewards, terminated) のタプル
            agent.memory.push(obs, actions, next_obs, rewards, terminated_flag)

            obs = next_obs
            episode_reward += sum(rewards.values())
            total_steps += 1

            if terminated_flag:
                done_flag = True

            # 4. 学習ステップ
            if len(agent.memory) > BATCH_SIZE * 5 and total_steps % LEARNING_FREQ == 0:
                batch = agent.memory.sample(BATCH_SIZE)
                loss = agent.learn(batch, TARGET_UPDATE_INTERVAL, total_steps)
                losses_history.append(loss)

            if done_flag:
                break

        rewards_history.append(episode_reward)

        # 進捗報告
        if i_episode % 100 == 0:
            avg_reward = np.mean(rewards_history[-100:])
            avg_loss = np.mean(losses_history[-100:]) if losses_history else 0.0
            print(f"Epi: {i_episode}/{NUM_EPISODES} | Steps: {t+1} | Total Steps: {total_steps} | Avg R (100): {avg_reward:.2f} | Avg Loss: {avg_loss:.4f} | Epsilon: {epsilon:.4f}")

    print("\n✅ 学習完了")

    # 5. 結果の可視化
    plt.figure(figsize=(12, 5))
    plt.plot(rewards_history)
    plt.title("QMIX Total Reward per Episode (Moving Avg 100)")
    plt.xlabel("Episode")
    plt.ylabel("Total Reward")
    plt.grid(True)
    plt.show()

run_qmix_training()

--- QMIX Learning Start --- (Agents: 2, Grid: 10x10)


TypeError: 'float' object is not subscriptable

In [ ]:
import torch
import numpy as np
import random
from collections import deque
import matplotlib.pyplot as plt
from typing import Dict, Tuple, List
# %debug
# ----------------------------------------------------
# 0. 環境設定と補助クラス (以前の定義を流用)
# ----------------------------------------------------

# 環境パラメータ (以前の定義に合わせてください)
GRID_SIZE = 10
NUM_AGENTS = 2
NUM_ORDERS = 3
PICKUP_LOCATIONS = [(1, 1), (8, 1), (5, 8)]
DROPOFF_LOCATION = (5, 5)

# QMIX学習パラメータ
BATCH_SIZE = 128
GAMMA = 0.99
EPS_START = 1.0
EPS_END = 0.05
EPS_DECAY = 50000        # 減衰をゆっくりにする (ステップ数ベース)
TARGET_UPDATE_INTERVAL = 200 # ターゲットネットワーク更新頻度 (ステップ数)
MEMORY_SIZE = 50000
NUM_EPISODES = 5000      # 実行するエピソード数
MAX_STEPS_PER_EPISODE = 200
LEARNING_FREQ = 4        # 4ステップごとに1回学習

# 環境・エージェントの形状定義
ACTION_SPACE = 5         # 5: 停止, 上, 下, 左, 右
OBS_SHAPE = 2 + 1 + NUM_AGENTS # 位置(2) + 荷物(1) + Agent ID(2) = 5
STATE_SHAPE = (2 + 1) * NUM_AGENTS + NUM_ORDERS # 位置(2) + 荷物(1)) * 2 + 注文(3) = 9

# --- 必須なクラス定義 (以前の回答から流用) ---
class WarehouseEnv:
    def __init__(self, size: int = GRID_SIZE, num_agents: int = NUM_AGENTS):
        self.size = size
        self.num_agents = num_agents
        self.action_space = 5  # 0:待機, 1:上, 2:下, 3:左, 4:右

        # Matplotlib用の図の保持
        self.fig = None
        self.ax = None

        # 状態の初期化
        self.reset()

    def reset(self) -> Dict[int, Tuple]:
        """環境を初期化し、初期状態を返します。"""
        self.agent_positions: Dict[int, Tuple[int, int]] = {
            i: (random.randint(0, self.size - 1), random.randint(0, self.size - 1))
            for i in range(self.num_agents)
        }
        self.agent_holding: Dict[int, bool] = {i: False for i in range(self.num_agents)}
        self.remaining_orders: List[int] = list(range(NUM_ORDERS))
        return self._get_obs()

    def _get_obs(self) -> Dict[int, Tuple]:
        obs = {}
        for i in range(self.num_agents):
            # obs[i] = (
            #     self.agent_positions[i],
            #     self.agent_holding[i],
            #     tuple(self.remaining_orders)
            # )
            other_agent_idx = 1 - i # 2エージェントの場合
            obs[i] = (
                self.agent_positions[i],      # 自分の位置
                self.agent_holding[i],        # 自分の状態
                self.agent_positions[other_agent_idx], # 相方の位置（重要！）
                tuple(self.remaining_orders)  # 残りの注文
            )
        return obs
    def step(self, actions: Dict[int, int]) -> Tuple[Dict, Dict, Dict, Dict]:
            next_positions: Dict[int, Tuple[int, int]] = {}
            # 1. 報酬とフラグの初期化
            rewards: Dict[int, float] = {i: 0.0 for i in range(self.num_agents)}
            picked_up_this_step = {i: False for i in range(self.num_agents)}
            delivered_this_step = {i: False for i in range(self.num_agents)}

            # 2. 位置の更新 (仮移動先の決定)
            for i, action in actions.items():
                current_x, current_y = self.agent_positions[i]
                next_x, next_y = current_x, current_y

                if action == 1: next_y += 1    # 上
                elif action == 2: next_y -= 1  # 下
                elif action == 3: next_x -= 1  # 左
                elif action == 4: next_x += 1  # 右

                next_x = np.clip(next_x, 0, self.size - 1)
                next_y = np.clip(next_y, 0, self.size - 1)
                next_positions[i] = (next_x, next_y)

            # 3. 衝突判定と移動の確定
            final_positions = self.agent_positions.copy()
            is_collision = False

            for i in range(self.num_agents):
                pos = next_positions[i]
                is_colliding = False
                for j in range(self.num_agents):
                    if i != j and pos == next_positions[j]:
                        is_colliding = True
                        break

                if is_colliding:
                    rewards[i] -= 5.0  # 衝突ペナルティ
                    is_collision = True
                    # 衝突した場合は元の位置から動かない(final_positionsを更新しない)
                else:
                    final_positions[i] = pos

            self.agent_positions = final_positions

            # 4. ピックアップ・ドロップオフ判定
            for i in range(self.num_agents):
                current_pos = self.agent_positions[i]

                # 荷物を持っていない場合：ピックアップ判定
                if not self.agent_holding[i]:
                    # 削除を安全に行うためリストのコピーで回す
                    for order_idx in list(self.remaining_orders):
                        if current_pos == PICKUP_LOCATIONS[order_idx]:
                            self.agent_holding[i] = True
                            self.remaining_orders.remove(order_idx)
                            picked_up_this_step[i] = True # フラグを立てる
                            break

                # 荷物を持っている場合：ドロップオフ判定
                elif self.agent_holding[i]:
                    if current_pos == DROPOFF_LOCATION:
                        self.agent_holding[i] = False
                        delivered_this_step[i] = True # フラグを立てる

            # 5. 報酬の最終集計（協調と積極性の強化）
            for i in range(self.num_agents):
                # (A) 時間経過による基本ペナルティ (「止まっていると損」と思わせる)
                rewards[i] -= 0.1

                # (B) 成果報酬：ピックアップ
                if picked_up_this_step[i]:
                    rewards[i] += 10.0  # チーム共通の成果
                    rewards[i] += 0.5   # 個別ボーナス（拾った本人へのインセンティブ）

                # (C) 成果報酬：ドロップオフ
                if delivered_this_step[i]:
                    rewards[i] += 50.0  # チーム共通の成果
                    rewards[i] += 2.0   # 個別ボーナス（届けた本人へのインセンティブ）

            # 全ての注文が完了したかチェック
            # 荷物を持っているエージェントがいないことも条件に加えるとより正確です
            is_all_delivered = len(self.remaining_orders) == 0 and not any(self.agent_holding.values())
            done = {i: is_all_delivered for i in range(self.num_agents)}

            return self._get_obs(), rewards, done, {"collision": is_collision}

    # --- 追加された可視化メソッド ---
    def render(self, mode='text', sleep_time=0.5):
        """
        環境を可視化します。
        mode='text': コンソールに文字で表示
        mode='graphic': Matplotlibで図として表示
        """
        if mode == 'text':
            self._render_text()
        elif mode == 'graphic':
            self._render_graphic(sleep_time)

    def _render_text(self):
        grid = [['.' for _ in range(self.size)] for _ in range(self.size)]

        # 場所のマーク (y座標は下から上へ増えるため、表示時は反転させるか注意が必要)
        # ここでは (0,0) を左下として扱います
        x, y = DROPOFF_LOCATION
        grid[self.size - 1 - y][x] = 'D'  # Dropoff

        for idx in self.remaining_orders:
            x, y = PICKUP_LOCATIONS[idx]
            grid[self.size - 1 - y][x] = 'P'  # Pickup

        for i, pos in self.agent_positions.items():
            x, y = pos
            char = f'A{i}'
            if self.agent_holding[i]:
                char = f'H{i}' # Holding

            # 同じ場所に重なった場合の表示処理（簡易）
            if grid[self.size - 1 - y][x] not in ['.', 'P', 'D']:
                grid[self.size - 1 - y][x] += char
            else:
                grid[self.size - 1 - y][x] = char

        print("-" * (self.size * 3))
        for row in grid:
            print(" ".join([f"{c:>2}" for c in row]))
        print("-" * (self.size * 3))

    # --- WarehouseEnv クラス内の _render_graphic メソッドの修正 ---
    # ⚠️ 注意: これは WarehouseEnv クラスの内部にあると仮定
    def _render_graphic(self, sleep_time):
        # sleep_time はここでは完全に無視される（run_learned_agent側で処理）

        if self.fig is None:
            plt.ioff() # インタラクティブモードをオフにする（描画更新はdisplayに任せる）
            self.fig, self.ax = plt.subplots(figsize=(6, 6))

        self.ax.clear()
        self.ax.set_xlim(-0.5, self.size - 0.5)
        self.ax.set_ylim(-0.5, self.size - 0.5)
        self.ax.set_xticks(range(self.size))
        self.ax.set_yticks(range(self.size))
        self.ax.grid(True)
        self.ax.set_title(f"Orders Remaining: {len(self.remaining_orders)}")

        # ドロップオフ地点 (赤色)
        dx, dy = DROPOFF_LOCATION
        self.ax.add_patch(patches.Rectangle((dx-0.5, dy-0.5), 1, 1, color='red', alpha=0.3, label='Dropoff'))
        self.ax.text(dx, dy, 'Drop', ha='center', va='center', fontsize=8, color='darkred')

        # ピックアップ地点 (青色)
        for idx in self.remaining_orders:
            px, py = PICKUP_LOCATIONS[idx]
            self.ax.add_patch(patches.Rectangle((px-0.5, py-0.5), 1, 1, color='blue', alpha=0.3, label='Pickup'))
            self.ax.text(px, py, 'Pick', ha='center', va='center', fontsize=8, color='darkblue')

        # エージェント (円)
        colors = ['green', 'orange', 'purple', 'cyan']
        for i, pos in self.agent_positions.items():
            ax, ay = pos
            color = colors[i % len(colors)]
            edgecolor = 'black'
            linewidth = 1
            if self.agent_holding[i]:
                linewidth = 3
                edgecolor = 'red' # 荷物持ち強調

            circle = patches.Circle((ax, ay), 0.3, facecolor=color, edgecolor=edgecolor, linewidth=linewidth, label=f'Agent {i}')
            self.ax.add_patch(circle)
            self.ax.text(ax, ay, f'A{i}', ha='center', va='center', color='white', fontweight='bold')

        # 🚨 描画更新と待機を削除！ (run_learned_agent側で処理する)
        # plt.draw()
        # if sleep_time > 0.0:
        #   plt.pause(sleep_time)

    def render_frame(self):
        fig, ax = plt.subplots(figsize=(6, 6))

        ax.set_xlim(-0.5, self.size - 0.5)
        ax.set_ylim(-0.5, self.size - 0.5)
        ax.set_xticks(range(self.size))
        ax.set_yticks(range(self.size))
        ax.grid(True)

        # Dropoff
        dx, dy = DROPOFF_LOCATION
        ax.add_patch(patches.Rectangle((dx-0.5, dy-0.5), 1, 1, color='red', alpha=0.3))

        # Pickup
        for idx in self.remaining_orders:
            px, py = PICKUP_LOCATIONS[idx]
            ax.add_patch(patches.Rectangle((px-0.5, py-0.5), 1, 1, color='blue', alpha=0.3))

        # Agents
        colors = ['green', 'orange', 'purple', 'cyan']
        for i, (x, y) in self.agent_positions.items():
            circle = patches.Circle((x, y), 0.3, color=colors[i % len(colors)])
            ax.add_patch(circle)
            ax.text(x, y, f'A{i}', ha='center', va='center', color='white')

        fig.canvas.draw()
        image = np.asarray(fig.canvas.buffer_rgba())[..., :3]

        plt.close(fig)
        return image

# class QMixReplayMemory: ... (以前のリプレイメモリ定義をここに配置)
class QMixReplayMemory:
    def __init__(self, capacity):
        self.memory = deque(maxlen=capacity)
    def push(self, state, action, next_state, reward, done):
        self.memory.append((state, action, next_state, reward, done))
    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)
    def __len__(self):
        return len(self.memory)

# class MLPAgent: ... (MLPAgentの定義をここに配置)
class MLPAgent(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(MLPAgent, self).__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        return self.net(x)

# class QMixer: ... (QMixerの定義をここに配置)
# 以前の回答から DeepQMixer を QMixer として定義したものを使用
class QMixer(torch.nn.Module):
    def __init__(self, num_agents, state_dim, hidden_dim, hypernet_embed_dim):
        super(QMixer, self).__init__()
        self.num_agents = num_agents
        self.state_dim = state_dim
        self.hidden_dim = hidden_dim

        # ハイパーネットワークの隠れ層サイズ
        hyper_net_hidden = 64

        # --- ハイパーネットワーク W1 (2層構造化) ---
        self.hyper_w1 = torch.nn.Sequential(
            torch.nn.Linear(state_dim, hyper_net_hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(hyper_net_hidden, hidden_dim * num_agents)
        )
        self.hyper_b1 = torch.nn.Linear(state_dim, hidden_dim)

        # --- ハイパーネットワーク W2 (2層構造化) ---
        self.hyper_w2 = torch.nn.Sequential(
            torch.nn.Linear(state_dim, hyper_net_hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(hyper_net_hidden, hidden_dim)
        )
        self.hyper_b2 = torch.nn.Sequential(
            torch.nn.Linear(state_dim, hyper_net_hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(hyper_net_hidden, 1)
        )

    def forward(self, agent_qs, states):
        batch_size = agent_qs.size(0)

        w1 = torch.abs(self.hyper_w1(states))
        w1 = w1.view(batch_size, self.num_agents, self.hidden_dim)

        agent_qs = agent_qs.view(batch_size, self.num_agents, 1)
        hidden = torch.bmm(agent_qs.transpose(1, 2), w1)

        b1 = self.hyper_b1(states).view(batch_size, 1, self.hidden_dim)
        hidden = F.elu(hidden + b1)

        w2 = torch.abs(self.hyper_w2(states))
        w2 = w2.view(batch_size, self.hidden_dim, 1)

        q_tot = torch.bmm(hidden, w2)

        b2 = self.hyper_b2(states).view(batch_size, 1, 1)

        q_tot = q_tot + b2
        return q_tot.squeeze(-1)

# class IntegratedQMixAgent: ... (前回の修正版クラス定義をここに配置)
class IntegratedQMixAgent:
    def __init__(self, env, obs_shape, state_shape, n_actions, lr=5e-4, gamma=0.99, mixing_embed_dim=32, hidden_dim=64, memory_capacity=50000):
        self.env = env
        self.n_agents = env.num_agents
        self.n_actions = n_actions
        self.gamma = gamma
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.memory = QMixReplayMemory(memory_capacity)
        self.agent_net = MLPAgent(obs_shape, hidden_dim, n_actions).to(self.device)
        self.mixer_net = QMixer(self.n_agents, state_shape, mixing_embed_dim, hypernet_embed_dim=hidden_dim).to(self.device)

        self.target_agent_net = MLPAgent(obs_shape, hidden_dim, n_actions).to(self.device)
        self.target_mixer_net = QMixer(self.n_agents, state_shape, mixing_embed_dim, hypernet_embed_dim=hidden_dim).to(self.device)

        self.target_agent_net.load_state_dict(self.agent_net.state_dict())
        self.target_mixer_net.load_state_dict(self.mixer_net.state_dict())

        params = list(self.agent_net.parameters()) + list(self.mixer_net.parameters())
        self.optimizer = torch.optim.Adam(params, lr=lr)

    def _obs_to_tensor(self, obs: Dict[int, Tuple], is_state: bool = False):
        if is_state:
            state_vec = []
            for i in range(self.n_agents):
                pos_tuple = obs[i][0]
                state_vec.extend([pos_tuple[0] / (GRID_SIZE - 1), pos_tuple[1] / (GRID_SIZE - 1)])
                state_vec.append(1.0 if obs[i][1] else 0.0)

            # 残り注文の処理
            remaining_orders_set = set(obs[0][2])
            for order_idx in range(NUM_ORDERS):
                state_vec.append(1.0 if order_idx in remaining_orders_set else 0.0)
            return torch.FloatTensor(state_vec).to(self.device).unsqueeze(0)

        else:
            tensors = {}
            for i in range(self.n_agents):
                # --- ここも修正：リスト内包表記ではなく明示的なインデックス指定に ---
                pos_tuple = obs[i][0]
                obs_i = [pos_tuple[0] / (GRID_SIZE - 1), pos_tuple[1] / (GRID_SIZE - 1)]

                obs_i.append(1.0 if obs[i][1] else 0.0)
                agent_id_vec = [0.0] * self.n_agents
                agent_id_vec[i] = 1.0
                obs_i.extend(agent_id_vec)
                tensors[i] = torch.FloatTensor(obs_i).to(self.device).unsqueeze(0)
            return tensors

    def get_actions(self, obs: Dict[int, Tuple], epsilon: float) -> Dict[int, int]:
        actions = {}
        if random.random() < epsilon:
            for i in range(self.n_agents):
                actions[i] = random.randint(0, self.n_actions - 1)
        else:
            agent_obs_tensors = self._obs_to_tensor(obs, is_state=False)
            with torch.no_grad():
                for i in range(self.n_agents):
                    q_values = self.agent_net(agent_obs_tensors[i])
                    actions[i] = q_values.max(dim=-1)[1].item()
        return actions

    def learn(self, batch, target_update_interval, update_counter):
        current_state = torch.cat([self._obs_to_tensor(t[0], is_state=True) for t in batch], dim=0)
        next_state = torch.cat([self._obs_to_tensor(t[3], is_state=True) for t in batch], dim=0)
        rewards = torch.FloatTensor([sum(t[2].values()) for t in batch]).to(self.device).unsqueeze(1)
        terminated = torch.FloatTensor([t[4] for t in batch]).to(self.device).unsqueeze(1)
        actions_batch = torch.LongTensor([[t[1][i] for i in range(self.n_agents)] for t in batch]).to(self.device)
        obs_batch = [torch.cat([self._obs_to_tensor(t[0], is_state=False)[i] for t in batch], dim=0) for i in range(self.n_agents)]
        next_obs_batch = [torch.cat([self._obs_to_tensor(t[3], is_state=False)[i] for t in batch], dim=0) for i in range(self.n_agents)]

        # 1. 現在のQ_totの計算
        agent_qs = []
        for i in range(self.n_agents):
            q_vals = self.agent_net(obs_batch[i])
            chosen_q = torch.gather(q_vals, dim=1, index=actions_batch[:, i].unsqueeze(1))
            agent_qs.append(chosen_q)

        agent_qs = torch.cat(agent_qs, dim=1)
        q_tot = self.mixer_net(agent_qs, current_state)

        # 2. TDターゲットの計算 (Double DQN 適用)
        target_agent_qs = []
        with torch.no_grad():
            for i in range(self.n_agents):
                next_q_online = self.agent_net(next_obs_batch[i])
                next_action_argmax = next_q_online.max(dim=1)[1].unsqueeze(1)

                target_q_vals = self.target_agent_net(next_obs_batch[i])

                target_max_q = target_q_vals.gather(dim=1, index=next_action_argmax)
                target_agent_qs.append(target_max_q)

            target_agent_qs = torch.cat(target_agent_qs, dim=1)
            target_q_tot = self.target_mixer_net(target_agent_qs, next_state)

        td_target = rewards + self.gamma * target_q_tot * (1 - terminated)

        # 3. 損失の計算と最適化
        loss = F.mse_loss(q_tot, td_target.detach())

        self.optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(self.agent_net.parameters(), 10)
        torch.nn.utils.clip_grad_norm_(self.mixer_net.parameters(), 10)

        self.optimizer.step()

        if update_counter % target_update_interval == 0:
            self.update_target_networks()

        return loss.item()

    def update_target_networks(self):
        self.target_agent_net.load_state_dict(self.agent_net.state_dict())
        self.target_mixer_net.load_state_dict(self.mixer_net.state_dict())

    def save_model(self, path="qmix_model.pth"):
        """モデルの重みとオプティマイザの状態を保存"""
        torch.save({
            'agent_net_state_dict': self.agent_net.state_dict(),
            'mixer_net_state_dict': self.mixer_net.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
        }, path)
        print(f"✅ Model saved to {path}")

    def load_model(self, path="qmix_model.pth"):
        """保存された状態を読み込み"""
        checkpoint = torch.load(path, map_location=self.device)
        self.agent_net.load_state_dict(checkpoint['agent_net_state_dict'])
        self.mixer_net.load_state_dict(checkpoint['mixer_net_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

        # ターゲットネットワークも同期させる
        self.update_target_networks()
        print(f"✅ Model loaded from {path}")
# ----------------------------------------------------
# 3. 学習ループ
# ----------------------------------------------------

def run_qmix_training():
    env = WarehouseEnv(GRID_SIZE, NUM_AGENTS)

    # リプレイメモリはここで定義されている
    replay_buffer = QMixReplayMemory(MEMORY_SIZE)

    agent = IntegratedQMixAgent(env, OBS_SHAPE, STATE_SHAPE, ACTION_SPACE, gamma=GAMMA, mixing_embed_dim=64, hidden_dim=128)

    total_steps = 0
    rewards_history = []
    losses_history = []

    print(f"--- QMIX Learning Start --- (Agents: {NUM_AGENTS}, Grid: {GRID_SIZE}x{GRID_SIZE})")

    for i_episode in range(1, NUM_EPISODES + 1):
        obs = env.reset()
        episode_reward = 0
        done_flag = False

        for t in range(MAX_STEPS_PER_EPISODE):
            # 1. ε-greedy法による行動選択
            epsilon = EPS_END + (EPS_START - EPS_END) * np.exp(-1. * total_steps / EPS_DECAY)
            actions = agent.get_actions(obs, epsilon)

            # 2. 環境ステップ
            next_obs, rewards, done, info = env.step(actions)
            terminated_flag = all(done.values())

            # チーム全体の終了フラグ
            terminated_flag = all(done.values())

            # 3. リプレイバッファに保存
            # リプレイメモリに保存されるデータは (obs, actions, next_obs, rewards, terminated) のタプル
            # agent.memory.push(obs, actions, next_obs, rewards, terminated_flag)
            # agent.memory.push(obs, actions, next_obs, rewards, terminated_flag)
            agent.memory.push(obs, actions, rewards, next_obs, terminated_flag)

            obs = next_obs
            episode_reward += sum(rewards.values())
            total_steps += 1

            if terminated_flag:
                done_flag = True

            # 4. 学習ステップ
            if len(agent.memory) > BATCH_SIZE * 5 and total_steps % LEARNING_FREQ == 0:
                batch = agent.memory.sample(BATCH_SIZE)
                loss = agent.learn(batch, TARGET_UPDATE_INTERVAL, total_steps)
                losses_history.append(loss)

            if done_flag:
                break

        rewards_history.append(episode_reward)

        # 進捗報告
        if i_episode % 100 == 0:
            avg_reward = np.mean(rewards_history[-100:])
            avg_loss = np.mean(losses_history[-100:]) if losses_history else 0.0
            print(f"Epi: {i_episode}/{NUM_EPISODES} | Steps: {t+1} | Total Steps: {total_steps} | Avg R (100): {avg_reward:.2f} | Avg Loss: {avg_loss:.4f} | Epsilon: {epsilon:.4f}")

    print("\n✅ 学習完了")

    # 5. 結果の可視化
    plt.figure(figsize=(12, 5))
    plt.plot(rewards_history)
    plt.title("QMIX Total Reward per Episode (Moving Avg 100)")
    plt.xlabel("Episode")
    plt.ylabel("Total Reward")
    plt.grid(True)
    plt.show()

    return agent

# ----------------------------------------------------
# 実行
# ----------------------------------------------------

if __name__ == '__main__':
    agent = run_qmix_training()

--- QMIX Learning Start --- (Agents: 2, Grid: 10x10)
Epi: 100/5000 | Steps: 172 | Total Steps: 19422 | Avg R (100): 39.21 | Avg Loss: 37.1453 | Epsilon: 0.6942
Epi: 200/5000 | Steps: 60 | Total Steps: 37874 | Avg R (100): 46.29 | Avg Loss: 45.4341 | Epsilon: 0.4954
Epi: 300/5000 | Steps: 200 | Total Steps: 54956 | Avg R (100): 85.25 | Avg Loss: 55.1538 | Epsilon: 0.3665
Epi: 400/5000 | Steps: 69 | Total Steps: 70682 | Avg R (100): 90.71 | Avg Loss: 59.0611 | Epsilon: 0.2811
Epi: 500/5000 | Steps: 200 | Total Steps: 87707 | Avg R (100): 72.47 | Avg Loss: 55.3612 | Epsilon: 0.2144
Epi: 600/5000 | Steps: 200 | Total Steps: 103548 | Avg R (100): 73.55 | Avg Loss: 61.9592 | Epsilon: 0.1698
Epi: 700/5000 | Steps: 200 | Total Steps: 119917 | Avg R (100): 65.53 | Avg Loss: 72.2896 | Epsilon: 0.1363
Epi: 800/5000 | Steps: 200 | Total Steps: 138104 | Avg R (100): 32.61 | Avg Loss: 66.2566 | Epsilon: 0.1100
Epi: 900/5000 | Steps: 200 | Total Steps: 154218 | Avg R (100): 87.87 | Avg Loss: 74.2912 

In [ ]:
def save_model(agent, path="qmix_model.pth"):
    """モデルの重みとオプティマイザの状態を保存"""
    torch.save({
        'agent_net_state_dict': agent.agent_net.state_dict(),
        'mixer_net_state_dict': agent.mixer_net.state_dict(),
        'optimizer_state_dict': agent.optimizer.state_dict(),
    }, path)
    print(f"✅ Model saved to {path}")

def load_model(agent, path="qmix_model.pth"):
    """保存された状態を読み込み"""
    checkpoint = torch.load(path, map_location=agent.device)
    agent.agent_net.load_state_dict(checkpoint['agent_net_state_dict'])
    agent.mixer_net.load_state_dict(checkpoint['mixer_net_state_dict'])
    agent.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    # ターゲットネットワークも同期させる
    agent.update_target_networks()
    print(f"✅ Model loaded from {path}")

save_model(agent, path="qmix_model.pth")

In [ ]:
from PIL import Image # インポート忘れに注意
import matplotlib.patches as patches  # これが必要です

def save_agent_behavior_gif(agent, env, filename="agent_behavior.gif", max_steps=200):
    frames = []
    obs = env.reset()

    print("🎬 Generating frames for GIF...")

    for t in range(max_steps):
        frame = env.render_frame()

        # --- ここを修正：もし numpy配列なら PIL画像に変換する ---
        if isinstance(frame, np.ndarray):
            # もし [0, 1] の範囲なら 255倍するなどの処理が必要な場合があります
            if frame.max() <= 1.0:
                frame = (frame * 255).astype(np.uint8)
            frame = Image.fromarray(frame)
        # --------------------------------------------------

        frames.append(frame)

        actions = agent.get_actions(obs, epsilon=0.0)
        next_obs, rewards, done, info = env.step(actions)
        obs = next_obs

        if all(done.values()):
            # 最後のフレーム処理
            last_frame = env.render_frame()
            if isinstance(last_frame, np.ndarray):
                last_frame = Image.fromarray((last_frame * 255).astype(np.uint8)) if last_frame.max() <= 1.0 else Image.fromarray(last_frame)
            frames.append(last_frame)
            print(f"✅ Goal reached in {t} steps!")
            break

    if frames:
        # frames[0] が確実に PIL Image になっているので save が使えます
        frames[0].save(
            filename,
            save_all=True,
            append_images=frames[1:],
            duration=200,
            loop=0
        )
        print(f"💾 GIF saved as {filename}")


env = WarehouseEnv(GRID_SIZE, NUM_AGENTS)
save_agent_behavior_gif(agent, env, "warehouse_qmix.gif")